## Parsing Playground

we can try some tools to parse pdf with a lot of images.
- Using Docling local vision model
- Using Vision model API services (e.g. Gemini vision, LLamaIndex)

Using local model is free, but limited to model size

Using on premise API service needs additional cost to implement, but provides better model to choose without hardware limitations

### Local

In [1]:
# install dependencies
%pip install "docling[vlm]" "transformers" "torch" "torchvision"

  Using cached filetype-1.2.0-py2.py3-none-any.whl.metadata (6.5 kB)
  Using cached pluggy-1.6.0-py3-none-any.whl.metadata (4.8 kB)
  Using cached beautifulsoup4-4.15.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached scipy-1.15.3-cp310-cp310-win_amd64.whl.metadata (60 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached referencing-0.37.0-py3-none-any.whl.metadata (2.8 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached mdurl-0.1.2-py3-none-any.whl.met

In [1]:
import os

os.environ["TORCH_COMPILE_DISABLE"] = "1"
from pypdf import PdfReader, PdfWriter
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions, 
    smolvlm_picture_description
)
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling_core.types.doc import PictureItem

def prepare_pdf_subset(input_pdf_path, output_pdf_path, num_pages):
    """
    Fungsi untuk mengambil n-halaman pertama dari PDF.
    Ini sangat penting untuk menghemat VRAM dan waktu proses saat testing.
    """
    reader = PdfReader(input_pdf_path)
    writer = PdfWriter()
    
    # Ambil halaman sesuai jumlah yang diminta (atau maksimal halaman yang ada)
    total_pages = len(reader.pages)
    pages_to_extract = min(num_pages, total_pages)
    
    for i in range(pages_to_extract):
        writer.add_page(reader.pages[i])
        
    with open(output_pdf_path, "wb") as f_out:
        writer.write(f_out)
        
    print(f"PDF berhasil dipotong menjadi {pages_to_extract} halaman pertama.")
    return output_pdf_path

def parse_pdf_to_unified_text(pdf_path, max_pages=5):
    # 1. Siapkan PDF sementara dengan jumlah halaman terbatas
    temp_pdf = "temp_subset.pdf"
    prepare_pdf_subset(pdf_path, temp_pdf, max_pages)
    
    # 2. Konfigurasi Pipeline Docling dengan SmolVLM
    pipeline_options = PdfPipelineOptions()
    pipeline_options.do_picture_description = True  
    pipeline_options.picture_description_options = smolvlm_picture_description
    
    # Prompt untuk fokus pada UI
    pipeline_options.picture_description_options.prompt = (
        "This is a screenshot of a web application interface. "
        "Describe the visible UI elements, buttons, menus, and the overall functionality shown in detail."
    )
    
    # 3. Inisialisasi Converter
    print("Memuat model visi (SmolVLM) ke VRAM... Mohon tunggu.")
    converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
        }
    )
    
    # 4. Mulai Proses Ekstraksi & Inferensi
    print(f"Mulai memproses teks dan gambar dari {temp_pdf}...")
    result = converter.convert(temp_pdf)
    doc = result.document
    
    # (Opsional) Cek di terminal/log jika gambar berhasil dianotasi
    image_count = sum(1 for item, _ in doc.iterate_items() if isinstance(item, PictureItem))
    print(f"Selesai! Ditemukan {image_count} gambar/screenshot pada {max_pages} halaman ini.")
    
    # 5. SATUKAN SEMUA MENJADI TEKS UTUH
    # export_to_markdown() secara otomatis merangkai paragraf PDF dan anotasi VLM
    unified_text = doc.export_to_markdown()
    
    # Bersihkan file PDF sementara
    if os.path.exists(temp_pdf):
        os.remove(temp_pdf)
        
    return unified_text

# ==========================================
# CARA PENGGUNAAN
# ==========================================
pdf_file = "./data/MODUL PEMBELAJARAN.pdf" # Ganti dengan path file Anda

# Parse hanya 5 halaman pertama
hasil_teks_utuh = parse_pdf_to_unified_text(pdf_file, max_pages=5)

print("\n\n=== HASIL TEKS BERSATU (MARKDOWN) ===\n")
print(hasil_teks_utuh)

c:\Users\Lenovo\anaconda3\envs\law\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PDF berhasil dipotong menjadi 5 halaman pertama.
Memuat model visi (SmolVLM) ke VRAM... Mohon tunggu.
Mulai memproses teks dan gambar dari temp_subset.pdf...


[transformers] Model config: pad_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 128002. This may result in unexpected behavior.
Loading weights: 100%|██████████| 471/471 [00:00<00:00, 2445.37it/s]
[INFO] 2026-08-11 00:01:38,508 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-08-11 00:01:38,523 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-08-11 00:01:38,551 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\Lenovo\anaconda3\envs\law\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.pth
[INFO] 2026-08-11 00:01:38,552 [RapidOCR] main.py:50: Using C:\Users\Lenovo\anaconda3\envs\law\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.pth
[INFO] 2026-08-11 00:01:38,785 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-08-11 00:01:38,787 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-08-11 00:01:38,792 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\Lenovo\anacond

KeyboardInterrupt: 

In [2]:
import os
# Pertahankan baris ini
os.environ["TORCH_COMPILE_DISABLE"] = "1"

from pypdf import PdfReader, PdfWriter
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions, 
    smolvlm_picture_description
)
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling_core.types.doc import PictureItem

def parse_pdf_to_unified_text(pdf_path, max_pages=5):
    temp_pdf = "temp_subset.pdf"
    
    # (Fungsi prepare_pdf_subset asumsikan sudah ada di atas seperti sebelumnya)
    prepare_pdf_subset(pdf_path, temp_pdf, max_pages)
    
    # 2. Konfigurasi Pipeline Docling
    pipeline_options = PdfPipelineOptions()
    pipeline_options.do_picture_description = True  
    pipeline_options.picture_description_options = smolvlm_picture_description
    pipeline_options.picture_description_options.prompt = (
        "This is a screenshot of a web application interface. "
        "Describe the visible UI elements, buttons, menus, and the overall functionality shown in detail."
    )
    
    # ==========================================
    # SOLUSI UNTUK std::bad_alloc (OUT OF MEMORY)
    # ==========================================
    pipeline_options.num_threads = 1  # Wajib: Paksa proses 1 halaman bergantian
    # ==========================================
    
    print("Memuat model visi (SmolVLM) ke VRAM... Mohon tunggu.")
    converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
        }
    )
    
    print(f"Mulai memproses teks dan gambar dari {temp_pdf}...")
    result = converter.convert(temp_pdf)
    doc = result.document
    
    unified_text = doc.export_to_markdown()
    
    if os.path.exists(temp_pdf):
        os.remove(temp_pdf)
        
    return unified_text

# ==========================================
# CARA PENGGUNAAN
# ==========================================
pdf_file = "./data/MODUL PEMBELAJARAN.pdf"

# Saran tambahan: Coba ubah max_pages=1 atau 2 dulu untuk tes pertama
hasil_teks_utuh = parse_pdf_to_unified_text(pdf_file, max_pages=2) 

print("\n\n=== HASIL TEKS BERSATU (MARKDOWN) ===\n")
print(hasil_teks_utuh)

PDF berhasil dipotong menjadi 2 halaman pertama.


ValueError: "PdfPipelineOptions" object has no field "num_threads"

In [3]:
import os
import gc  # Garbage Collector bawaan Python
# Wajib untuk menghindari error compiler C++
os.environ["TORCH_COMPILE_DISABLE"] = "1"

from pypdf import PdfReader, PdfWriter
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions, 
    smolvlm_picture_description
)
from docling.document_converter import DocumentConverter, PdfFormatOption

def parse_pdf_page_by_page(pdf_path, max_pages=5):
    # 1. Konfigurasi Docling (Tanpa num_threads yang error)
    pipeline_options = PdfPipelineOptions()
    pipeline_options.do_picture_description = True  
    pipeline_options.picture_description_options = smolvlm_picture_description
    pipeline_options.picture_description_options.prompt = (
        "This is a screenshot of a web application interface. "
        "Describe the visible UI elements, buttons, menus, and the overall functionality shown in detail."
    )
    
    print("Memuat model visi (SmolVLM) ke VRAM...")
    converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
        }
    )
    
    # 2. Baca PDF Asli
    reader = PdfReader(pdf_path)
    total_pages = min(max_pages, len(reader.pages))
    
    hasil_markdown_semua_halaman = []
    
    # 3. Proses LOOP: Satu Halaman Satu Waktu
    for i in range(total_pages):
        print(f"\n--- Memproses Halaman {i+1} dari {total_pages} ---")
        
        # Ekstrak Halaman ini ke PDF sementara
        temp_pdf = f"temp_page_{i}.pdf"
        writer = PdfWriter()
        writer.add_page(reader.pages[i])
        
        with open(temp_pdf, "wb") as f_out:
            writer.write(f_out)
            
        try:
            # Parse menggunakan Docling
            result = converter.convert(temp_pdf)
            doc = result.document
            
            # Ubah ke Markdown dan simpan ke list
            teks_halaman = doc.export_to_markdown()
            hasil_markdown_semua_halaman.append(teks_halaman)
            print(f"Sukses mengekstrak halaman {i+1}")
            
        except Exception as e:
            print(f"Gagal memproses halaman {i+1}. Error: {e}")
            
        finally:
            # Hapus PDF sementara
            if os.path.exists(temp_pdf):
                os.remove(temp_pdf)
                
            # ==================================================
            # PEMBERSIHAN MEMORI (Mencegah std::bad_alloc)
            # ==================================================
            gc.collect() # Bersihkan RAM CPU
            try:
                import torch
                if torch.cuda.is_available():
                    torch.cuda.empty_cache() # Bersihkan VRAM GPU
            except:
                pass

    # 4. Satukan semua teks Markdown dengan garis pemisah
    teks_utuh = "\n\n---\n\n".join(hasil_markdown_semua_halaman)
    return teks_utuh


# ==========================================
# CARA PENGGUNAAN
# ==========================================
pdf_file = "./data/MODUL PEMBELAJARAN.pdf"

# Mulai dengan max_pages=2 dulu untuk memastikan tidak ada error memori
hasil_teks_utuh = parse_pdf_page_by_page(pdf_file, max_pages=5)

print("\n\n=== HASIL TEKS BERSATU (MARKDOWN) ===\n")
print(hasil_teks_utuh)

Memuat model visi (SmolVLM) ke VRAM...

--- Memproses Halaman 1 dari 5 ---


Loading weights: 100%|██████████| 471/471 [00:00<00:00, 4510.95it/s]
[INFO] 2026-08-11 00:10:49,200 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-08-11 00:10:49,202 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-08-11 00:10:49,229 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\Lenovo\anaconda3\envs\law\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.pth
[INFO] 2026-08-11 00:10:49,230 [RapidOCR] main.py:50: Using C:\Users\Lenovo\anaconda3\envs\law\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.pth
[INFO] 2026-08-11 00:10:49,415 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-08-11 00:10:49,417 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-08-11 00:10:49,422 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\Lenovo\anaconda3\envs\law\Lib\site-packages\rapidocr\models\ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-08-11 00:10:49,422 [RapidOCR] main.py:50: Using C:\Users\Lenovo\anaconda3\

Sukses mengekstrak halaman 1

--- Memproses Halaman 2 dari 5 ---
Sukses mengekstrak halaman 2

--- Memproses Halaman 3 dari 5 ---
Sukses mengekstrak halaman 3

--- Memproses Halaman 4 dari 5 ---
Sukses mengekstrak halaman 4

--- Memproses Halaman 5 dari 5 ---
Sukses mengekstrak halaman 5


=== HASIL TEKS BERSATU (MARKDOWN) ===

## MODUL PEMBELAJARAN

## Accurate Online Accounting Software

<!-- image -->

The image displays a logo with a pink circle and the text "accurate" written in a light gray color. The circle is positioned on the left side of the text, and the text is positioned on the right side of the circle. The font used is a sans-serif font, which is easy to read and does not distract from the text.

The logo is simple and straightforward, with a clean and modern design. The use of a pink circle and the text "accurate" is a common design choice for logos, which is often used to convey a sense of professionalism and trustworthiness. The use of a sans-serif font is a common cho

This model runs about 11 minutes to parse 5 pages.

this model made some mispells on capturing the text. For example:
- "Mudah mengontrol ribuan stok dengan beragam satuan" but got: "Mudah mengentubi stok dengan beragam satu" 
- "Masuk Akun" but got: "Mausuk Akun,"

The possible reason for this is because we are using small size model, due to limitation to the hardware.

Mispels and wrong words can lead to false information. 

In [ ]:
%pip install llama-index llama-parse pypdf python-dotenv

  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached typing_inspect-0.9.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached mypy_extensions-1.1.0-py3-none-any.whl.metadata (1.1 kB)
   ---------------------------------------- 0.0/11.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.9 MB ? eta -:--:--
   ----------------

In [ ]:
import os
from dotenv import load_dotenv
from pypdf import PdfReader, PdfWriter
from llama_parse import LlamaParse

# 1. Muat API Key dari .env
load_dotenv()

# Verifikasi HANYA LlamaCloud API Key
if not os.environ.get("LLAMA_CLOUD_API_KEY"):
    raise ValueError("LLAMA_CLOUD_API_KEY tidak ditemukan di .env!")

def prepare_pdf_subset(input_pdf_path, output_pdf_path, num_pages):
    """
    Memotong n-halaman pertama dari PDF secara lokal.
    """
    reader = PdfReader(input_pdf_path)
    writer = PdfWriter()
    
    total_pages = len(reader.pages)
    pages_to_extract = min(num_pages, total_pages)
    
    for i in range(pages_to_extract):
        writer.add_page(reader.pages[i])
        
    with open(output_pdf_path, "wb") as f_out:
        writer.write(f_out)
        
    print(f"[Lokal] PDF berhasil dipotong menjadi {pages_to_extract} halaman.")
    return output_pdf_path

def parse_with_llamaparse_mode_a(pdf_path, max_pages=5):
    # 1. Siapkan PDF sementara
    temp_pdf = "temp_subset_llamaparse.pdf"
    prepare_pdf_subset(pdf_path, temp_pdf, max_pages)
    
    # 2. Konfigurasi LlamaParse (Versi A - Tanpa Vendor Eksternal)
    print("[Cloud] Mengirim dokumen ke LlamaParse (Engine Bawaan)...")
    
    instruksi_ui = (
        "Dokumen ini memuat panduan dengan banyak screenshot antarmuka web (UI). "
        "Harap ekstrak semua teks yang terlihat di dalam gambar/screenshot secara akurat "
        "dan pertahankan struktur dokumen (judul, paragraf, tabel) sebaik mungkin."
        "khusus tabel, buatkan ulang tabelnya dalam format markdown"
        "berikan deskripsi singkat gambarnya disertai awalan: 'Deskrpisi gambar:. . .(deskripsi)'."
        "sertakan juga bagian mana dari gambar yang di-highlight dengan panah atau kotak merah."
    )
    
    parser = LlamaParse(
        result_type="markdown",
        parsing_instruction=instruksi_ui,
        use_vendor_multimodal_model=False 
    )
    
    # 3. Eksekusi Parsing
    documents = parser.load_data(temp_pdf)
    
    # 4. Ambil dan GABUNGKAN teks dari semua halaman
    # (Ini adalah baris yang diperbaiki)
    unified_text = "\n\n".join([doc.text for doc in documents])
    
    # Bersihkan file lokal
    if os.path.exists(temp_pdf):
        os.remove(temp_pdf)
        
    return unified_text

# ==========================================
# CARA PENGGUNAAN
# ==========================================
pdf_file = "./data/MODUL PEMBELAJARAN.pdf" # Ganti dengan path dokumen Anda

# Jalankan ekstraksi untuk 5 halaman
hasil_markdown = parse_with_llamaparse_mode_a(pdf_file, max_pages=5)

print("\n\n=== HASIL TEKS BERSATU LLAMAPARSE (VERSI A) ===\n")
print(hasil_markdown)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_55868\1616050118.py:4: DeprecationWarning: The 'llama-parse' package is deprecated and will no longer receive updates. Please migrate to the new unified SDK. See https://developers.llamaindex.ai/python/cloud/llamaparse/getting_started/ and https://github.com/run-llama/llama-cloud-py/blob/main/README.md for migration instructions.
  from llama_parse import LlamaParse


[Lokal] PDF berhasil dipotong menjadi 5 halaman.
[Cloud] Mengirim dokumen ke LlamaParse (Engine Bawaan)...
Started parsing the file under job_id 334d02a7-12e8-4f30-9123-25e8ba9aa5dc


=== HASIL TEKS BERSATU LLAMAPARSE (VERSI A) ===

**MODUL PEMBELAJARAN**
Accurate Online Accounting Software

**Oleh:**
Product Enablement Consultant PT. Cipta Piranti Sejahtera

**Deskripsi gambar:** Halaman modul pembelajaran yang menampilkan judul "MODUL PEMBELAJARAN" di bagian atas, diikuti dengan teks "Accurate Online Accounting Software" dan informasi tentang penulisnya.



**Persiapan Accurate Online**

Sebelum memulai menggunakan Accurate Online, kita perlu melakukan beberapa langkah persiapan di halaman Persiapan Data Perusahaan, diantaranya:

1. **Akun Accurate Online**: membuat akun pengguna agar dapat mengakses Accurate Online.
2. **Data Perusahaan**: menginput dan melengkapi data perusahaan termasuk:
- Akun Kas/Bank
- Data Barang &#x26; Jasa
- Data Pelanggan
- Data Pemasok

**Cara Membuat Akun

In [ ]:
import os
from dotenv import load_dotenv
from llama_parse import LlamaParse

# 1. Muat API Key dari .env (Pastikan hanya ada LLAMA_CLOUD_API_KEY)
load_dotenv()

def parse_with_llamaparse_internal(pdf_path):
    print(f"[Cloud] Mengirim dokumen '{pdf_path}' ke LlamaParse (Engine Bawaan)...")
    
    # Instruksi tetap dipertahankan agar engine bawaan fokus mengekstrak teks pada UI
    instruksi_ui = (
        "Dokumen ini memuat panduan dengan banyak screenshot antarmuka web (UI). "
        "Harap ekstrak semua teks yang terlihat di dalam gambar/screenshot secara akurat "
        "dan pertahankan struktur dokumen (judul, paragraf, tabel)."
        "khusus tabel, buat ulang tabelnya dalam format markdown"
        "berikan deskripsi singkat gambarnya disertai awalan: 'Deskrpisi gambar:. . .(deskripsi)'."
        "sertakan juga bagian mana dari gambar yang di-highlight dengan panah atau kotak merah."
    )
    
    parser = LlamaParse(
        result_type="markdown",
        parsing_instruction=instruksi_ui,
        
        # --- MATIKAN PENGGUNAAN VENDOR API (OpenAI/Anthropic) ---
        use_vendor_multimodal_model=False 
    )
    
    # Langsung parse file PDF asli untuk semua halaman
    json_results = parser.get_json_result(pdf_path)
    
    # Ambil array "pages" dari data dokumen pertama
    pages_data = json_results[0]["pages"]
    
    hasil_akhir = []
    
    # Iterasi per halaman untuk menyisipkan penanda
    for page in pages_data:
        page_number = page["page"]
        page_text = page["text"]
        
        penanda = f"----halaman {page_number}----"
        teks_gabungan = f"{penanda}\n{page_text}"
        hasil_akhir.append(teks_gabungan)
        
    # Satukan seluruh halaman menjadi satu string dengan pemisah yang jelas
    unified_text = "\n\n\n".join(hasil_akhir)
        
    return unified_text

# ==========================================
# UJI COBA
# ==========================================
pdf_file = "./data/MODUL PEMBELAJARAN-13-16.pdf" # Pastikan path sesuai

# Eksekusi parsing untuk seluruh dokumen
hasil_markdown = parse_with_llamaparse_internal(pdf_file)

print("\n\n=== HASIL TEKS BERSATU DENGAN PENANDA HALAMAN ===\n")
print(hasil_markdown)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_17376\2336581985.py:3: DeprecationWarning: The 'llama-parse' package is deprecated and will no longer receive updates. Please migrate to the new unified SDK. See https://developers.llamaindex.ai/python/cloud/llamaparse/getting_started/ and https://github.com/run-llama/llama-cloud-py/blob/main/README.md for migration instructions.
  from llama_parse import LlamaParse


[Cloud] Mengirim dokumen './data/MODUL PEMBELAJARAN-13-16.pdf' ke LlamaParse (Engine Bawaan)...
Started parsing the file under job_id c38e71e5-82b5-428e-b5b5-be8a42d978c9


=== HASIL TEKS BERSATU DENGAN PENANDA HALAMAN ===

----halaman 1----
Kemudian akan muncul 3 tab informasi, yaitu:
 ●            Informasi umum : terkait nama akun, kode akun dsb.
 ●            Saldo  :  saldo awal akun sesuai dengan per tgl yang digunakan (Umum
              menggunakan tanggal mulai data).
 ●            Lain-lain : informasi tambahan mengenai catatan akun perkiraan
 accurate                                                       4           On Boarding 01
 V1212571-1540663                                                            Ezv Gunewan
 Dashboard           Persiapan Data Perusahaan Berita Akun Perkiraan                      4~
     E Data Baru x   Bank
E Informasi Umum     Saldo Lain-lain                                        Histori
 8   Saldo Awal
 B                   Nilai  Rp            

In [4]:
import os
from dotenv import load_dotenv
from llama_parse import LlamaParse

# 1. Muat API Key dari .env (Pastikan hanya ada LLAMA_CLOUD_API_KEY)
load_dotenv()

def parse_with_llamaparse_internal(pdf_path):
    print(f"[Cloud] Mengirim dokumen '{pdf_path}' ke LlamaParse (Engine Bawaan)...")
    
    # Instruksi tetap dipertahankan agar engine bawaan fokus mengekstrak teks pada UI
    instruksi_ui = (
        "Dokumen ini memuat panduan dengan banyak screenshot antarmuka web (UI) dan tabel. "
        "Harap ekstrak semua teks yang terlihat di dalam gambar/screenshot"
        "dan pertahankan struktur dokumen (judul, paragraf, tabel)."
        "SEMUA TABEL WAJIB DIUBAH KE FORMAT MARKDOWN TABLE (menggunakan garis pipa '|' dan header '---'). "
        "berikan deskripsi singkat gambarnya disertai awalan: 'Deskrpisi gambar:. . .(deskripsi)'."
        "sertakan juga bagian mana dari gambar yang di-highlight dengan panah atau kotak merah."
    )
    
    parser = LlamaParse(
        result_type="markdown",
        parsing_instruction=instruksi_ui,
        
        # --- MATIKAN PENGGUNAAN VENDOR API (OpenAI/Anthropic) ---
        use_vendor_multimodal_model=False 
    )
    
    # Langsung parse file PDF asli untuk semua halaman
    json_results = parser.get_json_result(pdf_path)
    
    # Ambil array "pages" dari data dokumen pertama
    pages_data = json_results[0]["pages"]
    
    hasil_akhir = []
    
    # Iterasi per halaman untuk menyisipkan penanda
    for page in pages_data:
        page_number = page["page"]
        page_text = page["text"]
        
        penanda = f"----halaman {page_number}----"
        teks_gabungan = f"{penanda}\n{page_text}"
        hasil_akhir.append(teks_gabungan)
        
    # Satukan seluruh halaman menjadi satu string dengan pemisah yang jelas
    unified_text = "\n\n\n".join(hasil_akhir)
        
    return unified_text

# ==========================================
# UJI COBA
# ==========================================
pdf_file = "./data/MODUL PEMBELAJARAN-13-16.pdf" # Pastikan path sesuai

# Eksekusi parsing untuk seluruh dokumen
hasil_markdown = parse_with_llamaparse_internal(pdf_file)

print("\n\n=== HASIL TEKS BERSATU DENGAN PENANDA HALAMAN ===\n")
print(hasil_markdown)

[Cloud] Mengirim dokumen './data/MODUL PEMBELAJARAN-13-16.pdf' ke LlamaParse (Engine Bawaan)...
Started parsing the file under job_id 1919f398-664f-4fd5-b700-2b539fe4081f


=== HASIL TEKS BERSATU DENGAN PENANDA HALAMAN ===

----halaman 1----
Kemudian akan muncul 3 tab informasi, yaitu:
 ●            Informasi umum : terkait nama akun, kode akun dsb.
 ●            Saldo  :  saldo awal akun sesuai dengan per tgl yang digunakan (Umum
              menggunakan tanggal mulai data).
 ●            Lain-lain : informasi tambahan mengenai catatan akun perkiraan
 accurate                                                       4           On Boarding 01
 V1212571-1540663                                                            Ezv Gunewan
 Dashboard           Persiapan Data Perusahaan Berita Akun Perkiraan                      4~
     E Data Baru x   Bank
E Informasi Umum     Saldo Lain-lain                                        Histori
 8   Saldo Awal
 B                   Nilai  Rp            

The model still failed to recreate the markdown table. probably, its because we use llma vision model (not very good compared to otehr smarter models)

## Gemini Vision (Multimodal)

In [5]:
%pip install google-generativeai pymupdf pillow python-dotenv

  Using cached google_generativeai-0.8.6-py3-none-any.whl.metadata (3.9 kB)
  Using cached google_ai_generativelanguage-0.6.15-py3-none-any.whl.metadata (5.7 kB)
  Using cached google_api_core-2.34.0-py3-none-any.whl.metadata (2.9 kB)
  Using cached google_api_python_client-2.198.0-py3-none-any.whl.metadata (7.0 kB)
  Using cached proto_plus-1.28.3-py3-none-any.whl.metadata (2.2 kB)
  Using cached protobuf-5.29.6-cp310-abi3-win_amd64.whl.metadata (592 bytes)
  Using cached googleapis_common_protos-1.75.1-py3-none-any.whl.metadata (8.5 kB)
INFO: pip is looking at multiple versions of google-api-core to determine which version is compatible with other requirements. This could take a while.
  Using cached google_api_core-2.33.0-py3-none-any.whl.metadata (3.2 kB)
INFO: pip is looking at multiple versions of google-api-core[grpc] to determine which version is compatible with other requirements. This could take a while.
  Using cached grpcio_status-1.83.0-py3-none-any.whl.metadata (1.2 kB)
I

In [7]:
import os
import fitz  # PyMuPDF
import google.generativeai as genai
from PIL import Image
from dotenv import load_dotenv

# 1. Muat API Key
load_dotenv()
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY tidak ditemukan di .env!")

# Konfigurasi SDK Gemini
genai.configure(api_key=GEMINI_API_KEY)

# Menggunakan Gemini 1.5 Flash (Sangat cepat dan andal untuk OCR & Vision)
model = genai.GenerativeModel('gemini-3.5-flash')

def parse_pdf_with_gemini(pdf_path):
    print(f"Membuka dokumen: {pdf_path}")
    
    # Buka PDF menggunakan PyMuPDF
    doc = fitz.open(pdf_path)
    hasil_akhir = []
    
    # Prompt yang ketat agar format sesuai kebutuhan RAG
    instruksi = (
        "Anda adalah asisten AI ahli ekstraksi data dokumen. "
        "Ubah gambar halaman dokumen ini menjadi teks Markdown yang sangat rapi.\n\n"
        "ATURAN KETAT:\n"
        "1. Teks Biasa: Ekstrak semua teks narasi dengan akurat. Hapus spasi ganda yang aneh akibat teks rata kanan-kiri (justified). "
        "Jadikan kalimat mengalir natural dengan spasi tunggal.\n"
        "2. Tabel: JIKA TERDAPAT TABEL, Anda WAJIB mengubahnya menjadi format Markdown Table (menggunakan | Kolom | Kolom |). "
        "Jika ada teks tabel yang terpotong/multibaris di gambar aslinya, gabungkan menjadi satu baris panjang yang rapi di format Markdown.\n"
        "3. Gambar/Screenshot UI: Jika ada screenshot antarmuka web, ekstrak teks yang ada di dalam gambar tersebut dan "
        "tambahkan deskripsi singkat mengenai fitur/tombol yang terlihat (misal: '[Screenshot UI menunjukkan tombol Submit dan menu Dropdown]').\n"
        "4. Jangan berikan kalimat pembuka/penutup (seperti 'Berikut hasilnya'), langsung berikan output Markdown-nya saja."
    )

    # Iterasi per halaman
    for page_num in range(len(doc)):
        print(f"Memproses halaman {page_num + 1} dari {len(doc)}...")
        
        # Ubah halaman PDF menjadi gambar dengan resolusi (DPI) 200 agar teks terbaca jelas
        page = doc.load_page(page_num)
        pix = page.get_pixmap(dpi=200)  
        
        # Konversi ke format yang bisa dibaca Gemini (PIL Image)
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        
        try:
            # Kirim instruksi dan gambar ke Gemini
            response = model.generate_content([instruksi, img])
            page_text = response.text.strip()
            
        except Exception as e:
            print(f"Gagal memproses halaman {page_num + 1}: {e}")
            page_text = "[GAGAL DIEKSTRAK OLEH LLM]"
            
        # Sisipkan penanda halaman persis seperti yang Anda minta
        penanda = f"----halaman {page_num + 1}----"
        teks_gabungan = f"{penanda}\n{page_text}"
        
        hasil_akhir.append(teks_gabungan)
        
    # Tutup dokumen PDF dari memori
    doc.close()
    
    # Satukan semua halaman
    unified_text = "\n\n\n".join(hasil_akhir)
    return unified_text

# ==========================================
# CARA PENGGUNAAN
# ==========================================
pdf_file = "./data/MODUL PEMBELAJARAN-13-16.pdf" # Sesuaikan dengan path Anda

hasil_markdown_gemini = parse_pdf_with_gemini(pdf_file)

print("\n\n=== HASIL EKSTRAKSI GEMINI VISION ===\n")
print(hasil_markdown_gemini)

Membuka dokumen: ./data/MODUL PEMBELAJARAN-13-16.pdf
Memproses halaman 1 dari 4...
Memproses halaman 2 dari 4...
Memproses halaman 3 dari 4...
Memproses halaman 4 dari 4...


=== HASIL EKSTRAKSI GEMINI VISION ===

----halaman 1----
Kemudian akan muncul 3 tab informasi, yaitu:
* **Informasi umum** : terkait nama akun, kode akun dsb.
* **Saldo** : saldo awal akun sesuai dengan per tgl yang digunakan (Umum menggunakan tanggal mulai data).
* **Lain-lain** : informasi tambahan mengenai catatan akun perkiraan

![Screenshot UI Accurate Online menampilkan halaman detail Akun Perkiraan pada tab 'Saldo', dengan field input 'Nilai' terisi Rp 30.000.000, 'per Tgl' terisi 31/12/2024, dan tombol simpan di sebelah kanan.]

### B. Barang dan Jasa

Persiapan Data Perusahaan selanjutnya adalah membuat data Barang & Jasa, dimana data ini dapat kita pergunakan untuk dijual maupun dibeli pada bisnis Anda. Untuk menambahkan data barang dan jasa klik tombol "Buat Data" pada laman persiapan data perusahaan.



Ini yang papa cari selama ini